# Checking the quality of the genetic data

Before testing associations with training outcomes, we checked whether the genetic measurements were suitable for analysis. Quality control removes measurements with too much missing information and identifies other features that may make a marker unsuitable for the planned comparison.

We used PLINK 1.9 to prepare a set of autosomal SNPs for each breed. Autosomes are the chromosomes other than the sex chromosomes; SNPs are markers representing a single DNA-letter difference. This chapter reports quality filtering and alignment of the retained markers to the reference genome. It does not yet assess genetic ancestry, relationships between dogs, or association with training outcomes.


## How we checked the data

We used the marker thresholds reported in the [2013TSA publication](https://www.nature.com/articles/s41598-022-18698-4) and a 10% missing-data cutoff for each dog, recalled by the investigator from that earlier analysis.

| Check | Cutoff | What it means |
| --- | --- | --- |
| Missing measurements per dog | No more than 10% | Exclude a dog if more than one in ten eligible markers lack a genotype. |
| Missing measurements per marker | No more than 10% | Exclude a marker if its genotype is missing for more than one in ten retained dogs. |
| Minor allele frequency | At least 2.5% | Require the less common genetic alternative to meet this frequency threshold. |
| Hardy–Weinberg equilibrium | Exclude p < 0.00001 | Exclude markers whose genotype proportions depart strongly from a simple population expectation. |

We examined the breeds separately. Within each breed, we first identified markers on autosomes with usable positions, unique identifiers and coordinates, and SNP alleles. Markers on other chromosomes, markers without usable positions, repeated coordinates, and insertion/deletion markers were set aside from this analysis. This does not imply that every marker set aside is erroneous.

We then checked missing measurements per dog, followed by missing measurements per marker, allele frequency, and Hardy–Weinberg equilibrium, in that order. Missingness and allele frequency were calculated using the broader Scheme B group. Hardy–Weinberg equilibrium was assessed using Scheme A controls. We applied the resulting marker set to both schemes so that their comparison would reflect the change in controls rather than a change in the markers tested.


## Dogs and markers retained

Every dog passed the missing-data cutoff. The highest proportion of missing measurements before sample filtering was 0.56% in German Shorthaired Pointers and 6.53% in Labrador Retrievers. These percentages were calculated across the eligible autosomal SNPs before marker-level statistical filtering.

| Filtering step | German Shorthaired Pointer markers | Labrador Retriever markers |
| --- | ---: | ---: |
| Starting array markers | 229,991 | 229,991 |
| Eligible autosomal SNPs | 220,807 | 220,807 |
| After the marker missing-data cutoff | 220,445 | 220,357 |
| After the allele-frequency cutoff | 168,905 | 171,572 |
| After the Hardy–Weinberg cutoff | 168,167 | 171,522 |

| Comparison | Breed | Controls | Behavioral cases | Total dogs | Retained markers |
| --- | --- | ---: | ---: | ---: | ---: |
| A: Validated or Certified controls | German Shorthaired Pointer | 96 | 9 | 105 | 168,167 |
| B: Broader control group | German Shorthaired Pointer | 107 | 9 | 116 | 168,167 |
| A: Validated or Certified controls | Labrador Retriever | 39 | 7 | 46 | 171,522 |
| B: Broader control group | Labrador Retriever | 41 | 7 | 48 | 171,522 |

PLINK completed without warnings. Every retained genotype was checked against the prepared input data, and no original input was changed. Within each breed, the two comparisons retain identical markers and behavioral cases.


## Aligning alleles to the reference genome

The study genotypes were supplied on CanFam3.1 using the CanineHD platform. Before phasing and imputation, we moved the retained marker coordinates to CanFam4, the assembly used by the Dog10K reference panel, and aligned their allele labels to that genome. This alignment was applied to both breeds and both definitions of the comparison groups.

An allele's position in a PLINK file does not establish its DNA strand. We used CanineHD sequence annotations and the reference genomes to determine how the recorded allele letters correspond to the reference. The shared non-ambiguous markers consistently supported Illumina TOP reporting, a convention based on surrounding DNA sequence. This convention guided interpretation of A/T and C/G markers, whose allele letters alone cannot distinguish the two strands.

### Markers retained after alignment

| Breed | Markers passing quality filters | Markers aligned to CanFam4 | A/T or C/G markers retained |
| --- | ---: | ---: | ---: |
| German Shorthaired Pointer | 168,167 | 165,064 | 8,202 |
| Labrador Retriever | 171,522 | 168,341 | 8,389 |

Within each breed, both comparison groups use the same aligned markers. All dogs were retained. Every retained genotype was checked against its original QC call, and an independent PLINK export agreed with every generated genotype.

Markers without sufficient strand information, a usable coordinate mapping, or compatible reference alleles were left out of the aligned dataset. They remain in the original QC files; inability to align a marker does not establish that its genotype measurement was wrong.

### What alignment allows us to do

The aligned genotypes provide inputs for reference-based phasing and imputation. Compatibility with the Dog10K panel and imputation accuracy will be assessed in the following chapters. Biological annotation on ROS_Cfam_1.0 is a separate step.

The structure analyses used the original QC marker set. Consistent changes to allele labels preserve genetic similarity among dogs; this alignment did not identify conflicting allele encodings between our comparison groups. The smaller, reference-aligned marker set serves a different purpose from the original QC baseline.

## What these checks establish

The resulting datasets provide an autosomal SNP baseline for the next stage of the study. Passing these filters does not establish every sample's identity, demonstrate that dogs are unrelated, or show that the study has enough information to estimate each association reliably. Those questions require further assessment.

The Hardy–Weinberg test can identify unusual genotype proportions, but a departure is not proof of laboratory error. Breed history and relationships among sampled dogs can affect its interpretation. The allele-frequency threshold also does not guarantee that there are enough copies of an allele among the small number of behavioral cases.

An initial coordinate comparison found matches for all 18 markers listed in the historical target reference. Twelve remained after filtering in each breed. All twelve retained targets in each breed remain available after reference alignment. Historical effect alleles and directions still need to be reconciled before interpreting replication results. A target unavailable after filtering is not evidence of failed replication.

Sex-chromosome analysis and genotype-based sex checks remain separate work. The next chapter will examine genetic structure and relationships among the dogs and assess which association models the retained data can support.


## Apply or review the genotype filters

The following calls the QC script with the agreed thresholds. It uses the active input manifest and a native PLINK 1.9 executable. A completed QC dataset is checked and reused. In a new working copy without QC outputs, the cell runs the filtering procedure; existing datasets are never deleted to force a rerun.

In [ ]:
from pathlib import Path
import sys

study_root = next(p for p in [Path.cwd(), *Path.cwd().parents]
                  if (p / "scripts/notebook_support.py").exists())
sys.path.insert(0, str(study_root / "scripts"))
from notebook_support import read_json, study_path, verify_datasets, run_script

In [ ]:
qc_path = study_root / "data/qc/manifest.private.json"
if not qc_path.exists():
    from notebook_support import plink_environment
    environment_file = plink_environment(study_root)
    run_script(study_root, "array_qc.py", "--root", study_root,
               "--environment", environment_file)
qc = read_json(qc_path)
assert qc["status"] == "complete"
verify_datasets(study_root, qc)
print("QC datasets verified:", len(qc["datasets"]))

### Retrieve reference resources for a fresh analysis

This optional cell downloads missing public resources without replacing those already present. Leave `download_missing_resources` false when reviewing the completed study. For a fresh analysis, set it true and run this cell before the alignment cell above. The genome downloads are substantial.

In [ ]:
download_missing_resources = False
if download_missing_resources:
    run_script(study_root, "download_imputation_assets.py")
    run_script(study_root, "download_reference_genomes.py")
    manifest_file = study_root / "data/reference/array-manifests/caninehd_b.csv"
    if not manifest_file.exists():
        from urllib.request import urlretrieve
        manifest_file.parent.mkdir(parents=True, exist_ok=True)
        urlretrieve("https://support.illumina.com/content/dam/illumina-support/documents/downloads/productfiles/caninehd/caninehd_b.csv", manifest_file)
else:
    print("Using existing reference resources.")

### Align retained markers to the reference

The alignment script uses the CanineHD sequence annotations, the CanFam3.1 and CanFam4 genomes, and the coordinate-conversion chain. If alignment has already completed, this cell verifies its datasets. For fresh alignment, the public resources and native liftOver tool must be available. The tool environment can be created from a WSL terminal with `bash scripts/setup_imputation_wsl.sh`. Original CanFam3.1 QC files are preserved.

In [ ]:
aligned_path = study_root / "data/aligned/manifest.private.json"
if not aligned_path.exists():
    resources = ["data/reference/array-manifests/caninehd_b.csv",
                 "data/reference/genomes/canFam3.2bit",
                 "data/reference/genomes/canFam4.2bit",
                 "data/reference/imputation-pilot/canFam3ToCanFam4.over.chain.gz"]
    missing = [p for p in resources if not study_path(study_root, p).exists()]
    if missing:
        raise FileNotFoundError("Prepare the reference resources listed below before alignment: " + ", ".join(missing))
    run_script(study_root, "reference_alignment.py")
alignment = read_json(aligned_path)
assert alignment["status"] == "complete"
verify_datasets(study_root, alignment)
print("Aligned datasets verified:", len(alignment["datasets"]))